# Week 5: Model Deployment & Training Efficiencies


In this lesson notebook we will apply this week's material to the family of GPT2 models with specific focus on memory consumption and model qualities.

**Note:** We should stress that using small (and older) models like GPT-2 is not necessarily representative of the effectiveness of the techniques for more recent models. Also, we are running only a few hundred steps for the training runs, obviously affecting the results. Hyperparameter tuning also wasn't done. **So the purpose of this notebook is to introduce and test the ideas, not to conduct a detailed comparison.**

We will use the Stanford Sentiment Treebank as a dataset to compare the models. This notebook also uses components and approaches of the '[Fine tuning Google Colab notebook](https://huggingface.co/blog/4bit-transformers-bitsandbytes)' discussing Bits and Bytes, and Hugging Face's notebook [text classification example](https://github.com/huggingface/notebooks/blob/main/examples/text_classification.ipynb) notebook.


Here is the structure of the lesson notebook and the points of interest:

0. Setup
1. Dataset Creation and Configuration: Sentiment Classification
2. The Base Model: GPT2-medium
    - Memory Consumption, pre- and during training
    - Fine-tuning Result
3. Quantization
    - Memory Consumption of 8-bit and 4-bit quantized models

4. LoRA: Fine-tuning with Few Parameters I
    - Memory Size during training compared to base models
    - Fine-tuning Result
    - Size of LoRA parameters
    - Saving and loading  

5. Soft Prompt Tuning: Fine-tuning with Few Parameters II
    - Memory Size during training compared to base models
    - Finetuning Result
    - Size of Soft Prompt parameters

6. QLoRA: Fine-tuning of GPT2-Large with Few Parameters & Aggressive Quantization
    - Memory Size during training compared to base and PEFT models
    - Fine-tuning Result

This notebook runs on a T4 processor.

**Note:** if you want to look at memory consumptions using the Resources tab, you may need to restart the session multiple times. If you do so, comment out the pip installs and rerun the Setup and Data Preparation sections. Then continue from where you want to continue.


##0. Setup


Installs & Imports:

In [ ]:
%%capture

!pip install -U -q datasets
#!pip install datasets==2.21.0
!pip install -U -q transformers
!pip install accelerate -U -q           # Quantization, Distribution
!pip install -U -q peft                  # LoRA
!pip install -U -q evaluate
!pip install -U -q bitsandbytes             # QLoRA

In [ ]:
import sys
import numpy as np
import torch

import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import BitsAndBytesConfig

from datasets import load_dataset
import evaluate

from peft import LoraConfig, TaskType, PeftModel, get_peft_model
from peft import load_peft_weights, set_peft_model_state_dict
from peft import PromptEncoderConfig, prepare_model_for_kbit_training

import datasets
import random
import pandas as pd
from IPython.display import display, HTML

import wandb
wandb.init(mode="disabled")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Some useful definitions (see Text Classification notebook):

In [ ]:
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, datasets.ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))


def preprocess_function(examples):
    return tokenizer(examples[sentence1_key], truncation=True)



def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

def show_currently_allocated_gpu_mem():
  torch.cuda.empty_cache()
  mem = torch.cuda.memory_allocated()
  print(f"Current GPU memory allocation (GB): {mem/1024**3}")

## 1. Data Setup

We use the GLUE dataset, loading the data for the Stanford Sentiment Treebank task. We will also right away define the tokenizer for our models (they all use the GPT2 tokenizer).

In [ ]:
task = actual_task = "sst2"
tokenizer_model_name = "gpt2-medium"  # GPT2 tokenizers hopefully are the same for all sizes. We pick this one.
batch_size = 16

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
dataset = load_dataset("nyu-mll/glue", actual_task)
metric = evaluate.load('glue', actual_task)


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

To access an actual element, you need to select a split first, then give an index:

In [ ]:
dataset["train"][2]

{'sentence': 'that loves its characters and communicates something rather beautiful about human nature ',
 'label': 1,
 'idx': 2}

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset.

In [ ]:
show_random_elements(dataset["train"])

,sentence,label,idx
0,monster truck-loving good ol' boys and peroxide blond honeys whose worldly knowledge comes from tv reruns and supermarket tabloids,negative,58012
1,is silly but strangely believable,positive,36286
2,the rotting underbelly of middle america,negative,31980
3,this rude and crude film does deliver a few gut-busting laughs,positive,45644
4,gooding offers a desperately ingratiating performance .,negative,5646
5,under-10 set,negative,24202
6,visual delights,positive,26203
7,"terminally bland ,",positive,37474
8,parker displays in freshening the play,positive,66705
9,over a late-inning twist that just does n't make sense,negative,23332


You can call its `compute` method with your predictions and labels directly and it will return a dictionary with the metric(s) value:

In [ ]:
fake_preds = np.random.randint(0, 2, size=(64,))
fake_labels = np.random.randint(0, 2, size=(64,))
metric.compute(predictions=fake_preds, references=fake_labels)

{'accuracy': 0.40625}

In [ ]:
sentence1_key, sentence2_key = ("sentence", None)

Following (https://github.com/huggingface/notebooks/blob/main/examples/text_classification.ipynb), we construct a properly formated (for the Trainer class) dataset using the pre-process function defined above:

In [ ]:
encoded_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

Lastly, we define for the future analysis the base model, the metric, and the key for the validation data in the encoded dataset:

In [ ]:
metric_name = "accuracy"
base_model_name = "gpt2-medium"
validation_key = "validation"

## 2. Base Models

Now we can perform fine-tuning of our base models. We start with GPT2-medium, a 355m parameter model.

In [ ]:
medium_model = AutoModelForSequenceClassification.from_pretrained("gpt2-medium", num_labels=2)
medium_model.config.pad_token_id = medium_model.config.eos_token_id

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2-medium
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Let's make sure it works:

In [ ]:
input_dict = tokenizer(['this is fun', 'this is nice'], return_tensors='pt')
input_dict['labels'] = torch.tensor([1, 0])
input_dict.to('cuda')

{'input_ids': tensor([[5661,  318, 1257],
        [5661,  318, 3621]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1],
        [1, 1, 1]], device='cuda:0'), 'labels': tensor([1, 0], device='cuda:0')}

In [ ]:
medium_model.to('cuda')
preds = medium_model(**input_dict)

preds.loss

tensor(3.5826, device='cuda:0', grad_fn=<NllLossBackward0>)

In [ ]:
medium_model.to('cuda')

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 1024)
    (wpe): Embedding(1024, 1024)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-23): 24 x GPT2Block(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=3072, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=1024)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=4096, nx=1024)
          (c_proj): Conv1D(nf=1024, nx=4096)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=1024, out_features=2, bias=False)
)

In [ ]:
medium_model(**tokenizer(['this is fun', 'this is nice'], return_tensors='pt').to('cuda')).keys()

odict_keys(['logits', 'past_key_values'])

In [ ]:
preds[0].mean()

tensor(3.5826, device='cuda:0', grad_fn=<MeanBackward0>)

Now look at the GPU memory consumption in the resource list! Is it as expected?

You can also look at the current gpu memory consumption in this way:



In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 1.3640727996826172


Now we first define the trainer arguments and then the actual trainer for the base model:

In [ ]:
args = TrainingArguments(
    f"full_{base_model_name}-finetuned-{task}",
    eval_strategy = "steps",
    eval_steps = 100,
    logging_strategy = "steps",
    logging_steps = 100,
    save_strategy = "no",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=1,
    max_steps=300,
    weight_decay=0.01,
    load_best_model_at_end=False,
    metric_for_best_model=metric_name,
    report_to="none",
    run_name="test1"
)

Then we just need to pass all of this along with our datasets to the `Trainer`:

In [ ]:
medium_trainer = Trainer(
    medium_model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


As stated in the referenced Text Classification notebook: "You might wonder why we pass along the `tokenizer` when we already preprocessed our data. This is because we will use it one last time to make all the samples we gather the same length by applying padding, which requires knowing the model's preferences regarding padding (to the left or right? with which token?)..."

Now we train the model simply through the `train` method. Note that we did not have to write our own training and testing loops, these are abstracted and taken care off by the `Trainer` class.

In [ ]:
medium_trainer.train()

Step,Training Loss,Validation Loss,Accuracy
100,0.735564,0.310670,0.883028
200,0.339948,0.251248,0.904817
300,0.257265,0.279186,0.912844


TrainOutput(global_step=300, training_loss=0.4442589251200358, metrics={'train_runtime': 137.612, 'train_samples_per_second': 34.881, 'train_steps_per_second': 2.18, 'total_flos': 291439861039104.0, 'train_loss': 0.4442589251200358, 'epoch': 0.07125890736342043})

Again, observe the memory consumption during training! Is it ~3-5x of the original amount?

Let us save the model to disc:



In [ ]:
medium_trainer.save_model("./medium_model_base")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
!ls -al ./medium_model_base

total 1389572
drwxr-xr-x 2 root root       4096 May 31 01:25 .
drwxr-xr-x 1 root root       4096 May 31 01:25 ..
-rw-r--r-- 1 root root       1078 May 31 01:25 config.json
-rw-r--r-- 1 root root 1419331144 May 31 01:25 model.safetensors
-rw-r--r-- 1 root root        326 May 31 01:25 tokenizer_config.json
-rw-r--r-- 1 root root    3557680 May 31 01:25 tokenizer.json
-rw-r--r-- 1 root root       5265 May 31 01:25 training_args.bin


In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.015801429748535


What if we delete the trainer?

In [ ]:
del medium_trainer

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.015801429748535


Ah! (It may take a few seconds before the memory is freed.)

##3. Quantization of the Medium Model

We will now load the medium model from disc, but we will quantize to 4-bit. Please consider the memory consumption.

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
loaded_medium_model_4_bit = AutoModelForSequenceClassification.from_pretrained("./medium_model_base",
                                                                      num_labels=2,
                                                                      quantization_config=quantization_config)

loaded_medium_model_4_bit(**tokenizer('this is fun', return_tensors='pt').to('cuda'))['logits']

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


tensor([[ 1.5273, -1.0337]], device='cuda:0', grad_fn=<IndexBackward0>)

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.374131679534912


How much memory is used now? Is it as expected?

##4. LoRA - Fine-tuning using Less Parameters I

We'll create a second GPT2-medium model which we will use for (some of) our PEFT trainings. This base model will **not** change, as we will only train adapters.


In [ ]:
peft_base_model = AutoModelForSequenceClassification.from_pretrained("gpt2-medium", num_labels=2)
peft_base_model.config.pad_token_id = peft_base_model.config.eos_token_id

peft_base_model.to('cuda')
peft_base_model(**tokenizer('this is fun', return_tensors='pt').to('cuda'))['logits']


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2-medium
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[ 1.9292, -7.6093]], device='cuda:0', grad_fn=<IndexBackward0>)

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 5.704316139221191


Memory increase expected?

Next, we need to define the LoRA configuration, particularly *r*:

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=10,
    lora_alpha=200,
    lora_dropout=0.1
)

This configuration is used to create a PEFT model, which is defined through a base model and the configuration:

In [ ]:
!pip install -U -q torchao
peft_lora_model = get_peft_model(peft_base_model, lora_config)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 102.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


How many trainable parameters do we have?

In [ ]:
peft_lora_model.print_trainable_parameters()

trainable params: 985,088 || all params: 355,810,304 || trainable%: 0.2769


In [ ]:
args = TrainingArguments(
    f"lora_{base_model_name}-finetuned-{task}",
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "no",
    logging_strategy = "steps",
    logging_steps = 100,
    learning_rate=1.2e-4,   # set higher than for base model!
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=1,
    max_steps=300,
    weight_decay=0.01,
    load_best_model_at_end=False,
    metric_for_best_model=metric_name,
)


peft_lora_trainer = Trainer(
    peft_lora_model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


In [ ]:
peft_lora_trainer.train()

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss,Accuracy
100,1.136203,0.758208,0.618119
200,0.567631,0.315858,0.875000
300,0.355527,0.315456,0.881881


TrainOutput(global_step=300, training_loss=0.6864536539713542, metrics={'train_runtime': 107.7536, 'train_samples_per_second': 44.546, 'train_steps_per_second': 2.784, 'total_flos': 292389517393920.0, 'train_loss': 0.6864536539713542, 'epoch': 0.07125890736342043})

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 3.0715551376342773


What about the memory consumption? Expected?



Great. Now what about saving and loading the LoRA (only!) parameters?

In [ ]:
peft_model_path = "./my_lora_model"

peft_lora_trainer.model.save_pretrained(peft_model_path)

Let's look at the size of the saved adapter:

In [ ]:
!ls -al my_lora_model/

total 3876
drwxr-xr-x 2 root root    4096 May 31 01:35 .
drwxr-xr-x 1 root root    4096 May 31 01:35 ..
-rw-r--r-- 1 root root    1058 May 31 01:35 adapter_config.json
-rw-r--r-- 1 root root 3946624 May 31 01:35 adapter_model.safetensors
-rw-r--r-- 1 root root    5146 May 31 01:35 README.md


Good. Now we will load and use the LoRA model... twice (to look at the incremental memory consumptions). Note that loading the model requires the original base model and the adapters.

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 3.0715551376342773


In [ ]:
loaded_peft_model_1 = PeftModel.from_pretrained(peft_base_model,
                                        peft_model_path,
                                        is_trainable=False)

loaded_peft_model_1(**tokenizer('this is fun', return_tensors='pt').to('cuda'))['logits']

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


tensor([[-0.7034, -3.1365]], device='cuda:0')

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 3.0752177238464355


Look at the memory consumption.


Good. Essentially no change. Why? What if instead we loaded another base model?

##4. Soft Prompt Tuning

We will now repeat the procedure, but will tune parameters in a Soft Prompt ('virtual token's):

In [ ]:
peft_prompt_config = PromptEncoderConfig(task_type=TaskType.SEQ_CLS,
                                         num_virtual_tokens=10,
                                         encoder_hidden_size=384)

In [ ]:
peft_prompt_model = get_peft_model(peft_base_model, peft_prompt_config)

peft_prompt_model.to('cuda')
peft_prompt_model(**tokenizer('this is fun', return_tensors='pt').to('cuda'))['logits']

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
[transformers] GPT2ForSequenceClassification will not detect padding tokens in `inputs_embeds`. Results may be unexpected if using padding tokens in conjunction with `inputs_embeds.`


tensor([[ 0.0185, -0.4343]], device='cuda:0', grad_fn=<IndexBackward0>)

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 3.105544090270996


Memory increase as expected?

In [ ]:
peft_prompt_model.print_trainable_parameters()

trainable params: 947,968 || all params: 356,756,224 || trainable%: 0.2657


In [ ]:
args = TrainingArguments(
    f"prompt_{base_model_name}-finetuned-{task}",
    eval_strategy = "steps",
    eval_steps = 100,
    logging_strategy = "steps",
    logging_steps = 100,
    save_strategy = "no",   # no saving of checkpoints
    learning_rate=8e-5,   # set higher than for base model!
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=1,
    max_steps=1000,
    weight_decay=0.01,
    load_best_model_at_end=False,
    metric_for_best_model=metric_name,
)

In [ ]:
peft_prompt_trainer = Trainer(
    peft_prompt_model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


Let's train!

In [ ]:
# annoying lines will show up nevertheless. What could they mean? Let's discuss...

peft_prompt_trainer.train()

Step,Training Loss,Validation Loss,Accuracy
100,0.647740,0.517000,0.739679
200,0.516112,0.451885,0.794725
300,0.426391,0.470290,0.824541
400,0.418246,0.400487,0.825688
500,0.412503,0.385281,0.852064
600,0.398142,0.371273,0.850917
700,0.368025,0.369995,0.860092
800,0.379228,0.340352,0.868119
900,0.359932,0.340419,0.865826
1000,0.356730,0.339201,0.863532


TrainOutput(global_step=1000, training_loss=0.4283051071166992, metrics={'train_runtime': 431.8876, 'train_samples_per_second': 37.047, 'train_steps_per_second': 2.315, 'total_flos': 991218516295680.0, 'train_loss': 0.4283051071166992, 'epoch': 0.2375296912114014})

What about the size upon saving?

In [ ]:
peft_prompt_model_path = "./my_prompt_model"

peft_prompt_trainer.model.save_pretrained(peft_prompt_model_path)

In [ ]:
!ls -al my_prompt_model/

total 72
drwxr-xr-x 2 root root  4096 May 31 01:45 .
drwxr-xr-x 1 root root  4096 May 31 01:45 ..
-rw-r--r-- 1 root root   546 May 31 01:45 adapter_config.json
-rw-r--r-- 1 root root 49360 May 31 01:45 adapter_model.safetensors
-rw-r--r-- 1 root root  5124 May 31 01:45 README.md


In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 3.1145730018615723


##5. QLoRA

Now let's use QLoRA to fine-tune a model that is quantized down to 4 bits. We first need to specify the BitsAndBytes configuration, then the LoRA adapter, and then we'll train as always. But now we will use the XL model with 1.5bn parameters? That would **not** fit into our T4 chip for training purposes. Will it work with QLoRA? And how good will the results be?

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
qlora_model = AutoModelForSequenceClassification.from_pretrained("gpt2-XL", quantization_config=bnb_config, device_map={"":0})

qlora_model.config.pad_token_id = qlora_model.config.eos_token_id

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2-XL
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
qlora_model(**tokenizer('this is fun', return_tensors='pt').to('cuda'))['logits']

tensor([[ 0.6512, -0.7251]], device='cuda:0', grad_fn=<IndexBackward0>)

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.167047500610352


We need to do a few more adjustments:

In [ ]:
qlora_model.gradient_checkpointing_enable()
qlora_model = prepare_model_for_kbit_training(qlora_model)


In [ ]:
config = LoraConfig(
    r=10,
    lora_alpha=32,
    #target_modules=["query_key_value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

qlora_model = get_peft_model(qlora_model, config)
qlora_model.print_trainable_parameters()

trainable params: 3,075,200 || all params: 1,560,689,600 || trainable%: 0.1970


In [ ]:
#qlora_model.to('cuda')
qlora_model(**tokenizer('this is fun', return_tensors='pt').to('cuda'))['logits']

tensor([[ 0.6512, -0.7251]], device='cuda:0', grad_fn=<IndexBackward0>)

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.200601577758789


In [ ]:
args = TrainingArguments(
    f"qlora_gpt2-XL-finetuned-{task}",
    eval_strategy = "steps",
    eval_steps = 100,
    save_strategy = "no",
    logging_strategy = "steps",
    logging_steps = 100,
    learning_rate=1.2e-4,   # set higher than for base model!
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=1,
    max_steps=300,
    weight_decay=0.01,
    load_best_model_at_end=False,
    metric_for_best_model=metric_name,
    #push_to_hub=True,
)

qlora_trainer = Trainer(
    qlora_model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset[validation_key],
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)


In [ ]:
qlora_trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy
100,0.506739,0.365004,0.885321
200,0.301870,0.249864,0.918578
300,0.232134,0.251001,0.922018


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=300, training_loss=0.3469142405192057, metrics={'train_runtime': 982.9824, 'train_samples_per_second': 4.883, 'train_steps_per_second': 0.305, 'total_flos': 1425456276480000.0, 'train_loss': 0.3469142405192057, 'epoch': 0.07125890736342043})

In [ ]:
peft_qlora_model_path = "./my_qlora_model"

qlora_trainer.model.save_pretrained(peft_qlora_model_path)

In [ ]:
!ls -al my_qlora_model/

total 12048
drwxr-xr-x 2 root root     4096 May 31 02:05 .
drwxr-xr-x 1 root root     4096 May 31 02:05 ..
-rw-r--r-- 1 root root     1023 May 31 02:05 adapter_config.json
-rw-r--r-- 1 root root 12313312 May 31 02:05 adapter_model.safetensors
-rw-r--r-- 1 root root     5138 May 31 02:05 README.md


In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.2245283126831055


In [ ]:
del qlora_trainer

In [ ]:
show_currently_allocated_gpu_mem()

Current GPU memory allocation (GB): 4.2245283126831055


That's it! We hope that this gives you a good overview of all of the various approaches that allow you to deploy and train a model in much more efficient ways compared to 'training all parameters at full precision'.